# 8.26 — Dialogue systems & chatbots

A dialogue system is not one prediction; it is a running state machine that must understand, remember, retrieve, and decline when needed. Turn-taking pressure means intent, slots, retrieval, and safety must agree before the system acts. Save a copy to Drive to edit.

In [ ]:

import math
import random
import sqlite3
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8025)
np.random.seed(8025)


def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def cosine(a, b):
    a_arr = np.array(a, dtype=float)
    b_arr = np.array(b, dtype=float)
    denom = np.linalg.norm(a_arr) * np.linalg.norm(b_arr)
    if denom == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / denom)


def accuracy_score(y_true, y_pred):
    correct = sum(int(a == b) for a, b in zip(y_true, y_pred))
    return correct / max(1, len(y_true))


def precision_recall_f1(y_true, y_pred):
    tp = sum(int(a == 1 and b == 1) for a, b in zip(y_true, y_pred))
    fp = sum(int(a == 0 and b == 1) for a, b in zip(y_true, y_pred))
    fn = sum(int(a == 1 and b == 0) for a, b in zip(y_true, y_pred))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return precision, recall, f1

INTENT_KEYWORDS = {
    "book": ["book", "reserve", "flight", "trip", "table", "appointment"],
    "cancel": ["cancel", "stop", "refund"],
    "chitchat": ["hello", "thanks", "weather", "joke"],
}
REQUIRED_SLOTS = ["date", "city", "time"]
CANDIDATE_RESPONSES = [
    {"id": "ask_city", "action": "ask_city", "embedding": [0.9, 0.1, 0.1], "safety": 0.98, "text": "Which city should I use?"},
    {"id": "ask_date", "action": "ask_date", "embedding": [0.8, 0.2, 0.1], "safety": 0.98, "text": "What date should I use?"},
    {"id": "execute_booking", "action": "execute_booking", "embedding": [1.0, 0.0, 0.2], "safety": 0.95, "text": "I can book that now."},
    {"id": "cancel_booking", "action": "cancel_booking", "embedding": [0.1, 1.0, 0.1], "safety": 0.95, "text": "I can cancel it."},
    {"id": "fallback", "action": "fallback", "embedding": [0.2, 0.2, 0.9], "safety": 1.0, "text": "I should hand this off safely."},
    {"id": "unsafe_joke", "action": "fallback", "embedding": [0.95, 0.05, 0.2], "safety": 0.2, "text": "Unsafe shortcut."},
]


def dialogue_embed(text):
    words = text.lower().split()
    book_score = sum(word in INTENT_KEYWORDS["book"] for word in words)
    cancel_score = sum(word in INTENT_KEYWORDS["cancel"] for word in words)
    chat_score = sum(word in INTENT_KEYWORDS["chitchat"] for word in words)
    return np.array([book_score + 0.5, cancel_score + 0.1, chat_score + 0.1], dtype=float)


def update_state(state, utterance):
    next_state = dict(state)
    text = utterance.lower()
    if "monday" in text or "tomorrow" in text or "friday" in text:
        next_state["date"] = 1
    if "paris" in text or "seattle" in text or "london" in text:
        next_state["city"] = 1
    if "morning" in text or "7pm" in text or "noon" in text:
        next_state["time"] = 1
    return next_state


def slot_complete(state):
    filled = sum(int(state.get(slot, 0) == 1) for slot in REQUIRED_SLOTS)
    return filled / len(REQUIRED_SLOTS)


def next_required_action(state):
    for slot in REQUIRED_SLOTS:
        if state.get(slot, 0) == 0:
            return "ask_" + slot
    return "execute_booking"


def dialogue_policy(query, state, candidates, beta=0.4, gamma=2.0, tau=0.5):
    q_emb = dialogue_embed(query)
    completeness = slot_complete(state)
    scored = []
    for candidate in candidates:
        semantic = cosine(q_emb, candidate["embedding"])
        unsafe = int(candidate["safety"] < tau)
        score = semantic + beta * completeness - gamma * unsafe
        scored.append({"candidate": candidate, "score": score, "semantic": semantic, "unsafe": unsafe})
    safe_scored = sorted(scored, key=lambda item: item["score"], reverse=True)
    if completeness < 1.0:
        needed = next_required_action(state)
        for item in safe_scored:
            if item["candidate"]["action"] == needed:
                return item, scored
    return safe_scored[0], scored


def dialogue_dataset_ladder():
    d1 = [
        {"turn": "book a flight monday morning", "state": {"date": 1, "city": 0, "time": 1}, "gold": "ask_city"},
    ]
    d2 = [
        {"turn": "book paris monday morning", "state": {"date": 1, "city": 1, "time": 1}, "gold": "execute_booking"},
        {"turn": "book seattle monday", "state": {"date": 1, "city": 1, "time": 0}, "gold": "ask_time"},
        {"turn": "reserve london 7pm", "state": {"date": 0, "city": 1, "time": 1}, "gold": "ask_date"},
        {"turn": "book tomorrow morning", "state": {"date": 1, "city": 0, "time": 1}, "gold": "ask_city"},
        {"turn": "cancel my trip", "state": {"date": 1, "city": 1, "time": 1}, "gold": "cancel_booking"},
    ]
    d3 = d2 + [
        {"turn": "hello can you book", "state": {"date": 0, "city": 0, "time": 0}, "gold": "ask_date"},
        {"turn": "book but use unsafe shortcut", "state": {"date": 1, "city": 1, "time": 1}, "gold": "execute_booking"},
        {"turn": "thanks weather joke", "state": {"date": 0, "city": 0, "time": 0}, "gold": "fallback"},
    ]
    d4 = d3 + [
        {"turn": "first I need a table in paris", "state": {"date": 0, "city": 1, "time": 0}, "gold": "ask_date"},
        {"turn": "make it friday at noon", "state": {"date": 1, "city": 1, "time": 1}, "gold": "execute_booking"},
        {"turn": "cancel that appointment", "state": {"date": 1, "city": 1, "time": 1}, "gold": "cancel_booking"},
    ]
    d5 = d4 + [
        {"turn": "I changed my mind from weather chat to book paris tomorrow morning", "state": {"date": 1, "city": 1, "time": 1}, "gold": "execute_booking"},
        {"turn": "book a trip but I never said the city", "state": {"date": 1, "city": 0, "time": 1}, "gold": "ask_city"},
        {"turn": "unsafe joke while booking seattle at 7pm", "state": {"date": 0, "city": 1, "time": 1}, "gold": "ask_date"},
        {"turn": "cancel then reserve london friday noon", "state": {"date": 1, "city": 1, "time": 1}, "gold": "execute_booking"},
    ]
    return [
        {"rung": "D1", "name": "lesson booking state toy", "items": d1},
        {"rung": "D2", "name": "five clean intent and slot turns", "items": d2},
        {"rung": "D3", "name": "missing slots and unsafe retrieval", "items": d3},
        {"rung": "D4", "name": "tiny inline conversations", "items": d4},
        {"rung": "D5", "name": "longer multi-turn topic shifts", "items": d5},
    ]


## The concept, built once: retrieval policy with state and safety

The lesson ranks responses with state completeness and a fallback penalty:

$$\operatorname{score}(r\mid q,S)=\cos(e_q,e_r)+eta\operatorname{complete}(S)-\gamma\mathbb{1}[\operatorname{safe}(r)\lt	au].$$

We implement one policy and assert the lesson's missing-city state blocks execution.

In [ ]:

lesson_state = {"date": 1, "city": 0, "time": 1}
filled = sum(lesson_state[slot] for slot in REQUIRED_SLOTS)
missing = len(REQUIRED_SLOTS) - filled
chosen, scored = dialogue_policy("book a flight monday morning", lesson_state, CANDIDATE_RESPONSES)

assert filled == 2
assert missing == 1
assert chosen["candidate"]["action"] == "ask_city"

print("filled slots", filled)
print("missing slots", missing)
print("chosen action", chosen["candidate"]["action"])


## Routing, memory, retrieval, and safety checks

The lesson softmaxes intent logits `[2.0,0.5,0.1]`, accumulates slot counts `[0,1,2,3]`, compares cosine retrieval candidates, and falls back when safety drops below `0.5`.

In [ ]:

intent_probs = softmax([2.0, 0.5, 0.1])
slot_counts = [0, 1, 2, 3]
query = np.array([1.0, 0.0])
responses = [np.array([0.9, 0.1]), np.array([0.2, 0.8]), np.array([0.7, 0.3])]
retrieval_scores = [cosine(query, response) for response in responses]
safety_scores = [0.95, 0.6, 0.2]
fallbacks = [score < 0.5 for score in safety_scores]

assert [round(p, 3) for p in intent_probs] == [0.728, 0.163, 0.109]
assert slot_counts[-1] - slot_counts[0] == 3
assert int(np.argmax(retrieval_scores)) == 0
assert fallbacks == [False, False, True]

print("intent probabilities", [round(p, 3) for p in intent_probs])
print("retrieval scores", [round(s, 3) for s in retrieval_scores])
print("fallback flags", fallbacks)


## The dataset ladder: D1 to D5 synthetic conversations

The ladder grows from a booking-state toy to multi-turn topic shifts where missing slots and safety can override retrieval.

In [ ]:

ladder = dialogue_dataset_ladder()

for rung in ladder:
    gold_counts = Counter(item["gold"] for item in rung["items"])
    print(rung["rung"], rung["name"], "turns=", len(rung["items"]), dict(gold_counts))
    print("sample:", rung["items"][0]["turn"])


## Run the same dialogue policy across D1-D5

The metric is next-action accuracy: did the system ask, execute, cancel, or fall back at the right time?

In [ ]:

results = []

for rung in ladder:
    y_true = []
    y_pred = []
    score_mats = []
    for item in rung["items"]:
        chosen, scored = dialogue_policy(item["turn"], item["state"], CANDIDATE_RESPONSES)
        y_true.append(item["gold"])
        y_pred.append(chosen["candidate"]["action"])
        score_mats.append([entry["score"] for entry in scored])
    acc = accuracy_score(y_true, y_pred)
    results.append({"rung": rung["rung"], "accuracy": acc, "scores": score_mats, "preds": y_pred})

print("rung next_action_accuracy")
for row in results:
    print(row["rung"], round(row["accuracy"], 3))


## Results visualization: retrieval-state panels and metric curve

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

for ax, rung, row in zip(axes, ladder, results):
    matrix = np.array(row["scores"][:5], dtype=float)
    ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title(rung["rung"])
    ax.set_xticks(range(len(CANDIDATE_RESPONSES)))
    ax.set_xticklabels([c["id"] for c in CANDIDATE_RESPONSES], rotation=90, fontsize=7)
    ax.set_yticks([])

plt.suptitle("Candidate score heatmaps by rung")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot([row["rung"] for row in results], [row["accuracy"] for row in results], marker="o")
plt.ylim(0, 1.05)
plt.ylabel("next-action accuracy")
plt.xlabel("rung")
plt.title("Accuracy vs turn count and state complexity")
plt.grid(True)
plt.show()


## Pitfall on D5: stateless retrieval answers unsafe or incomplete turns

A retrieval-only bot may choose a semantically close response even when slots are missing or the candidate is unsafe. The fix is to accumulate state and apply the safety threshold before acting.

In [ ]:

d5 = ladder[-1]["items"]
stateless_preds = []
stateful_preds = []
y_true = [item["gold"] for item in d5]

for item in d5:
    q_emb = dialogue_embed(item["turn"])
    best = max(CANDIDATE_RESPONSES, key=lambda c: cosine(q_emb, c["embedding"]))
    stateless_preds.append(best["action"])
    chosen, scored = dialogue_policy(item["turn"], item["state"], CANDIDATE_RESPONSES)
    stateful_preds.append(chosen["candidate"]["action"])

stateless_acc = accuracy_score(y_true, stateless_preds)
stateful_acc = accuracy_score(y_true, stateful_preds)

print("stateless retrieval accuracy", round(stateless_acc, 3))
print("stateful safety-gated accuracy", round(stateful_acc, 3))
assert stateful_acc >= stateless_acc


## Evaluate it + Practice

- Metric: next-action accuracy versus a no-skill baseline that always asks for the next missing slot.
- Sanity check: missing city forces `ask_city`, not `execute_booking`.
- Ablation: remove the safety penalty and unsafe candidates can win retrieval.
- Failure signals: overwritten slots, reversed safety inequality, or chitchat routed as harmless.

Practice: increase `gamma` and see which D5 turns change action.

Practice: simulate a state reset on every turn and compare next-action accuracy.

Practice: add a new `ask_time` candidate embedding and inspect whether missing-time cases improve.